🐍 Design Patterns in Python

What are they?

Software design patterns are well-tested, reusable approaches to solving common problems in software engineering. Rather than providing complete, ready-made code, they serve as flexible solution templates that developers can adapt to different applications, technologies, and design requirements. Because these patterns have been applied, evaluated, and refined across many projects, they capture proven practices that can improve the structure, maintainability, and scalability of software. Using design patterns can also save development time by giving programmers a reliable starting point instead of requiring them to create a new solution for every recurring problem. In this way, design patterns are not only reusable solutions but also practical, optimized approaches that encourage clearer and more efficient software design.

## 1. Creational Design Patterns

Creational patterns handle how objects are created. They help make object creation flexible and separate it from the code that uses those objects.

- Singleton
- Builder
- Abstract Factory
- Factory Method
- Prototype

### - Singleton Pattern
The Singleton pattern ensures that a class has only one instance and provides a global point of access to it.

#### Singleton Example 1: Overriding __new__ on the Model
The __new__ method is responsible for instantiating the object before Pydantic initializes its fields. Storing a reference to the instance on a private class attribute ensures only one instance is ever created.

In [26]:
# Only one instance of AppSettings will be created, 
# and any subsequent attempts to create a new instance will return 
# the existing instance.
from typing import Any, ClassVar, Optional
from pydantic import BaseModel

# Singleton pattern implementation using Pydantic's BaseModel
class AppSettings(BaseModel):
    app_name: str = "MyApp"
    debug: bool = False

    # Class variable to hold the singleton instance
    _instance: ClassVar[Optional["AppSettings"]] = None

    # Override the __new__ method to control instance creation
    def __new__(cls, *args: Any, **kwargs: Any) -> "AppSettings":
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

# Example usage
settings1 = AppSettings(app_name="MyApp", debug=True)
settings2 = AppSettings(app_name="AnotherApp", debug=False)
# Both settings1 and settings2 refer to the same instance 
# because of the singleton pattern, only the first initialization will take effect.
print(settings1 is settings2)  # True
print(settings2.app_name)  # Output: MyApp


True
AnotherApp


#### Singleton Example 2: Module-Level Singleton with Pydantic BaseSettings
In modern Python and Pydantic applications, the idiomatic way to achieve a singleton configuration is using a module-level cached instance via functools.lru_cache.

This example uses functools.lru_cache to cache the result of get_settings(). The first call creates a ServerSettings instance; later calls return that same instance, providing a simple, module-level singleton pattern for accessing application settings.

In [27]:
# Make sure pydantic-settings is installed: uv add pydantic-settings
from enum import StrEnum
from functools import lru_cache
from pydantic_settings import BaseSettings

class Environment(StrEnum):
    PRODUCTION = "production"
    TEST = "test"
    DEVELOPMENT = "development"

class ServerSettings(BaseSettings):
    env: Environment = Environment.PRODUCTION
    max_connections: int = 100

# Use lru_cache to ensure that only one instance of ServerSettings is created and reused.
# The decorator caches the result of the function, so subsequent calls to get_settings() 
# will return the same instance.
@lru_cache()
def get_settings() -> ServerSettings:
    return ServerSettings()

# Example usage
s1 = get_settings()
s2 = get_settings()

# Both s1 and s2 refer to the same instance because of the lru_cache decorator.
print(s1 is s2)  # True
print(s1.env)  # production
print(s1.env is Environment.PRODUCTION)  # True

True
production
True


#### Example 3: Pydantic Singleton Logger Class
This logger maintains a single shared configuration (log level, target file path, and log count) across the application using a metaclass singleton.

In [28]:
# Enum gives log levels a fixed set of valid values instead of accepting arbitrary strings.
from enum import Enum
import logging
from threading import Lock
from typing import Any, ClassVar, Optional
from pydantic import BaseModel, PrivateAttr

class LogLevel(str, Enum):
    # Each enum member is also a string, which makes it convenient to pass to logging.
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"
    DEBUG = "DEBUG"

class SingletonMeta(type(BaseModel)):
    """Metaclass that returns one shared instance of each class that uses it."""
    # This dictionary maps each class to its one shared instance.
    # Any is used here because static type checkers do not always model metaclass
    # instances as ordinary class objects.
    _instances: ClassVar[dict[Any, Any]] = {}
    # A lock makes checking and creating the instance safe if threads call it together.
    _lock: ClassVar[Lock] = Lock()

    def __call__(cls: "SingletonMeta", *args: Any, **kwargs: Any) -> Any:
        # Python calls this method whenever someone constructs a class using this metaclass.
        with SingletonMeta._lock:
            # The first construction creates and stores the instance.
            if cls not in SingletonMeta._instances:
                # super().__call__ performs normal Pydantic model construction,
                # including calling model_post_init().
                SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
            # Later constructions return the stored instance; their arguments are ignored.
            return SingletonMeta._instances[cls]

class AppLogger(BaseModel, metaclass=SingletonMeta):
    # These are the public configuration values supplied at first construction.
    log_file: str = "app.log"
    default_level: LogLevel = LogLevel.INFO

    # PrivateAttr keeps runtime state out of Pydantic's regular model fields.
    # All references to this singleton share the same counter and logger object.
    _log_count: int = PrivateAttr(default=0)
    _logger: logging.Logger = PrivateAttr()

    def model_post_init(self, __context: Any) -> None:
        # Pydantic calls this after the model fields have been initialized.
        # Set the logger's threshold from the shared configuration.
        self._logger = logging.Logger("AppLogger", level=self.default_level.value)
        # FileHandler writes log records to the configured file (appending by default).
        handler = logging.FileHandler(self.log_file, encoding="utf-8")
        # Formatter controls how each line appears in the log file.
        formatter = logging.Formatter(
            "[%(asctime)s] %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
        )
        handler.setFormatter(formatter)
        self._logger.addHandler(handler)

    def log(self, message: str, level: Optional[LogLevel] = None) -> None:
        # If a call does not specify a level, use the singleton's configured default.
        effective_level = level or self.default_level
        # logging expects a numeric level; the enum's string name maps to that value.
        self._logger.log(getattr(logging, effective_level.value), message)
        # Increment the shared count for each call to log().
        self._log_count += 1

    @property
    def total_logs(self) -> int:
        # A property lets callers read the count without changing it directly.
        return self._log_count

# --- Usage Example ---
# The first call creates the singleton and sets its shared configuration.
logger_a = AppLogger(log_file="app.log", default_level=LogLevel.INFO)
# This call returns the same object; its different arguments do not replace the first configuration.
logger_b = AppLogger(log_file="different.log", default_level=LogLevel.ERROR)

# Both references write through the same logger and contribute to the same count.
logger_a.log("Starting application...")
logger_b.log("An error occurred!", LogLevel.ERROR)

# Confirm that construction returned the same object and retained the first configuration.
print(f"Is same instance? {logger_a is logger_b}")  # True
print(f"Shared log file: {logger_b.log_file}")  # app.log
print(f"Shared default level: {logger_b.default_level.value}")  # INFO
print(f"Total logs captured across references: {logger_a.total_logs}")  # 2

Is same instance? True
Shared log file: app.log
Shared default level: INFO
Total logs captured across references: 2


## 2. Structural Design Patterns

Structural patterns describe how classes and objects are composed to form larger structures. They help organize relationships between components while keeping systems flexible and easier to maintain.

- Adapter
- Bridge
- Composite
- Decorator
- Facade
- Flyweight
- Proxy

## 3. Behavioral Design Patterns

Behavioral patterns describe how objects communicate and share responsibilities. They help organize interactions so components can work together without becoming tightly coupled.

- Chain of Responsibility
- Command
- Interpreter
- Iterator
- Mediator
- Memento
- Observer
- State
- Strategy
- Template Method
- Visitor